<a href="https://colab.research.google.com/github/UdaraChamidu/EyeDoc/blob/main/Conversational_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Eye Disease Chat Bot** with Langchain and OpenAI LLM

In [8]:
!pip install langchain -qU
!pip install langchain-openai -qU      # LLM and Embedding model
!pip install langchain-chroma -qU      # vector database
!pip install langchain_community -qU

In [2]:
import os
from google.colab import userdata

# Initialize OpenAI LLM

In [3]:
from langchain_openai import ChatOpenAI

# Set OpenAI API key
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Initialize the ChatOpenAI model (language model)
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0 # randomness of the response
)

# Initialize Embedding Model

In [4]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### Load PDF Document

In [5]:
!pip install pypdf -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 5.5 MB/s eta 0:00:00


In [9]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF document
loader = PyPDFLoader("Kanski’s clinical ophthalmology _ a systematic approach.pdf")

docs = loader.load() # document object (page_content and metadata)

In [10]:
len(docs)

897

In [13]:
docs[500]

Document(metadata={'producer': '3-Heights(TM) PDF Producer 4.4.43.3 (http://www.pdf-tools.com); modified using iTextSharp 5.2.1 (c) 1T3XT BVBA', 'creator': 'Elsevier', 'creationdate': '2015-05-13T17:49:48+07:00', 'author': 'Brad Bowling', 'moddate': '2015-07-05T16:31:24+09:30', 'subject': "Kanski's Clinical Ophthalmology, Eighth Edition (2016) ii. doi:10.1016/B978-0-7020-5572-0.00025-8", 'title': "Kanski's Clinical Ophthalmology, Eighth Edition (2016)", 'source': 'Kanski’s clinical ophthalmology _ a systematic approach.pdf', 'total_pages': 897, 'page': 500, 'page_label': '489'}, page_content='CHAPTER\nOcular tumours 489\n12\nInvestigation\nExamination is sufficient for diagnosis in the majority of cases.\n• FA is of limited diagnostic value because there is no \npathognomonic pattern. The most common findings are an \nintrinsic tumour (‘dual’) circulation (Fig. 12.29A), mottled \nfluorescence during the arteriovenous phase and late diffuse \nleakage and staining. FA may, however, be us

### Split Documents into Chunks

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# Split the documents into chunks
# create splitter object
splits = text_splitter.split_documents(docs)

In [15]:
len(splits)

6524

In [17]:
splits[6000]

Document(metadata={'producer': '3-Heights(TM) PDF Producer 4.4.43.3 (http://www.pdf-tools.com); modified using iTextSharp 5.2.1 (c) 1T3XT BVBA', 'creator': 'Elsevier', 'creationdate': '2015-05-13T17:49:48+07:00', 'author': 'Brad Bowling', 'moddate': '2015-07-05T16:31:24+09:30', 'subject': "Kanski's Clinical Ophthalmology, Eighth Edition (2016) ii. doi:10.1016/B978-0-7020-5572-0.00025-8", 'title': "Kanski's Clinical Ophthalmology, Eighth Edition (2016)", 'source': 'Kanski’s clinical ophthalmology _ a systematic approach.pdf', 'total_pages': 897, 'page': 828, 'page_label': '817'}, page_content='drome, and in men may cause hypogonadism, impotence, sterility, \ndecreased libido, and occasionally gynaecomastia and galactor-\nrhoea. Up to 95% remain as microadenomas, though those that \ndo not may present initially with visual features.\nCorticotrophic adenoma\nA corticotrophic adenoma (a variant of basophil adenoma) \nsecretes ACTH and causes Cushing disease (Cushing syndrome')

### Create Vector Store and Retriever

In [18]:
from langchain_chroma import Chroma

# Create a vector store from the document chunks
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
# embedding vectors save in the vectorstore

In [19]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

# when user ask question, retrever gives appropriate chunks from vector store.

### Define Prompt Template

In [20]:
from langchain_core.prompts import ChatPromptTemplate

# Define the system prompt
system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"  # this context come from retriever
    # best chunks for the asked question from the vector space.
    # we can define number of chunks that need to come here from retriever
)

# Create the prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt), # 2 roles, system and human
        ("human", "{input}"), # user input
    ]
)

# after that final prompt created

In [21]:
prompt
# according to the user asked question, context will vary.

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

### Create Retrieval-Augmented Generation (RAG) Chain

In [22]:
# the rag chain contains QA chain and retriever

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt) # create QA chain using llm and prompt
# prompt = system prompt + user input
# system prompt = instruction + context

# Create the RAG chain containing QA chain and retriever
rag_chain = create_retrieval_chain(retriever, qa_chain)
# QA chain has to give answers to questions that ask by user

### Invoke RAG Chain with Example Questions

In [23]:
response = rag_chain.invoke({"input": "what is efficientNet ?"}) # ask questions
response["answer"]

"I'm sorry, but the information provided does not include details about EfficientNet. Therefore, I don't have specific information about EfficientNet in this context."

In [24]:
response = rag_chain.invoke({"input": "what is Deep learning"})
response["answer"]

'Deep learning is a type of machine learning that uses artificial neural networks to model and solve complex problems. It is a subset of machine learning that focuses on learning representations of data through multiple layers of abstraction. Deep learning algorithms are designed to automatically learn and improve from experience without being explicitly programmed. They have been used in various fields, including computer vision, natural language processing, and speech recognition.'

In [25]:
response = rag_chain.invoke({"input": "what is CNN"})
response["answer"]

'CNN can stand for different things depending on the context. In the medical field, CNN could refer to Computed Tomographic Venography, which is a fast high-resolution imaging technique used to visualize blood vessels. However, in a more general context, CNN often stands for Cable News Network, a popular news channel. If you have a specific context in mind, please provide more details for a more accurate answer.'

In [26]:
response = rag_chain.invoke({"input": "What is RAG architecture"})
response["answer"]

"I'm sorry, but based on the context provided, I don't have information about RAG architecture."

In [28]:
response = rag_chain.invoke({"input": "Give me list of eye diseases"})
response["answer"]

'Here is a list of eye diseases:\n\n1. Retinal venous occlusive disease\n2. Retinal arterial occlusive disease\n3. Ocular ischaemic syndrome\n4. Hypertensive eye disease\n5. Sickle cell retinopathy\n6. Lens\n7. Uveitis\n8. Retina\n9. Optic nerve\n10. Thyroid eye disease\n11. Infections\n12. Non-infective inflammatory disease\n13. Non-neoplastic vascular abnormalities\n14. Erysipelas\n15. Necrotizing fasciitis\n16. Viral infections\n17. Molluscum contagiosum\n18. Herpes zoster ophthalmicus\n19. Herpes simplex\n20. Blepharitis\n21. Chronic blepharitis\n22. Phthiriasis palpebrarum\n23. Tick infestation of the eyelid\n24. Angular blepharitis\n25. Childhood blepharokeratoconjunctivitis\n26. Ptosis\n\nThese are some of the eye diseases mentioned in the context provided.'

# **Add Chat History**

# Create a retriever that aware history

In [30]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# Define the contextualize system prompt
contextualize_system_prompt = (
    "using chat history and the latest user question, just reformulate question if needed and otherwise return it as is"
)

# Create the contextualize prompt template
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create the history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

#### Create History-Aware RAG Chain

**Define the prompt Template**

In [31]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

prompt

# now => ChatPromptTemplate(input_variables=['chat_history', 'context', 'input']

ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.

In [32]:
# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the history aware RAG chain
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

#### Manage Chat Session History

In [33]:
# different user has different history
# here we use session id. according to session id, store the history in a list

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Initialize the store for session histories (a dictionary)
store = {}

# Function to get the session history for a given session ID
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create the conversational RAG chain with session history
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# we can save the dictionary in jason format, save in a database.

#### Invoke Conversational RAG Chain with Example Questions

In [34]:
response = conversational_rag_chain.invoke(
    {"input": "what is diabitic retinopathy ?"},
    config={"configurable": {"session_id": "101"}},  # this is session id.
)   # each user have unique session id
response["answer"]

'Diabetic retinopathy is a complication of diabetes that affects the eyes. It is caused by damage to the blood vessels of the light-sensitive tissue at the back of the eye (retina). This condition can lead to vision problems and even blindness if left untreated.'

In [35]:
response = conversational_rag_chain.invoke(
    {"input": "what is Eye disease classification ?"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'Eye diseases can be classified in various ways based on different criteria. One common classification is based on the affected part of the eye, such as the cornea, retina, or uvea. Another classification is based on the underlying cause of the disease, such as genetic factors, infections, or systemic conditions like diabetes. Additionally, eye diseases can be classified as congenital or acquired, primary or secondary, and based on specific symptoms or signs.'

In [36]:
response = conversational_rag_chain.invoke(
    {"input": "eye diseases"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

"There are numerous eye diseases that can affect different parts of the eye and have various causes. Some common eye diseases include:\n\n1. Diabetic Retinopathy\n2. Glaucoma\n3. Cataracts\n4. Age-related Macular Degeneration (AMD)\n5. Retinal Detachment\n6. Conjunctivitis (Pink Eye)\n7. Keratitis\n8. Uveitis\n9. Retinitis Pigmentosa\n10. Blepharitis\n\nThese are just a few examples of the many eye diseases that can impact vision and eye health. It's essential to seek professional medical advice if you suspect you have an eye condition."

In [37]:
response = conversational_rag_chain.invoke(
    {"input": "can you list down eye diseases"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

"Certainly! Here is a list of various eye diseases:\n\n1. Diabetic Retinopathy\n2. Glaucoma\n3. Cataracts\n4. Age-related Macular Degeneration (AMD)\n5. Retinal Detachment\n6. Conjunctivitis (Pink Eye)\n7. Keratitis\n8. Uveitis\n9. Retinitis Pigmentosa\n10. Blepharitis\n11. Retinal Venous Occlusive Disease\n12. Retinal Arterial Occlusive Disease\n13. Ocular Ischaemic Syndrome\n14. Hypertensive Eye Disease\n15. Sickle Cell Retinopathy\n16. Thalassaemia Retinopathy\n17. Retinopathy of Prematurity\n18. Retinal Artery Macroaneurysm\n19. Primary Retinal Telangiectasia\n\nThese are just a few examples of eye diseases. It's important to consult with an eye care professional for proper diagnosis and treatment if you suspect you have an eye condition."

In [38]:
response = conversational_rag_chain.invoke(
    {"input": "explain "},
    config={"configurable": {"session_id": "101"}},  # previous session id
)
response["answer"]

"Certainly! Here is a brief explanation of some of the eye diseases listed:\n\n1. Diabetic Retinopathy: Damage to the blood vessels in the retina due to diabetes.\n2. Glaucoma: A group of eye conditions that damage the optic nerve, often due to high pressure in the eye.\n3. Cataracts: Clouding of the lens in the eye, leading to blurry vision.\n4. Age-related Macular Degeneration (AMD): Degeneration of the macula, leading to central vision loss.\n5. Retinal Detachment: Separation of the retina from its underlying layers, causing vision loss.\n6. Conjunctivitis (Pink Eye): Inflammation of the conjunctiva, causing redness and irritation.\n7. Keratitis: Inflammation of the cornea, often due to infection or injury.\n8. Uveitis: Inflammation of the uvea, the middle layer of the eye.\n9. Retinitis Pigmentosa: Inherited disorder causing progressive vision loss.\n10. Blepharitis: Inflammation of the eyelids, causing redness and irritation.\n\nThese eye diseases vary in causes, symptoms, and tre

In [39]:
response = conversational_rag_chain.invoke(
    {"input": "explain more"},
    config={"configurable": {"session_id": "102"}},  # for different session id
)
response["answer"]

'The Spiral of Tillaux is an important anatomical landmark used in surgery. It is an imaginary line that connects the insertions of the four recti muscles of the eye. These insertions are located progressively further away from the limbus in a spiral pattern. The closest insertion to the limbus is the medial rectus, followed by the inferior rectus, lateral rectus, and superior rectus.\n\nDuring surgery, this spiral pattern helps surgeons identify the correct locations for incisions and other procedures involving the eye muscles. By following the Spiral of Tillaux, surgeons can navigate the anatomy of the eye more effectively and reduce the risk of complications.\n\nIn summary, the Spiral of Tillaux is a useful guide for surgeons when performing eye surgery, helping them locate the insertions of the recti muscles accurately and safely.'

In [40]:
response = conversational_rag_chain.invoke(
    {"input": "My eyes feel dry and irritated all the time. Could this be a sign of something serious?"},
    config={"configurable": {"session_id": "102"}},
)
response["answer"]

'Dry and irritated eyes can be caused by various factors, including environmental conditions, prolonged screen time, certain medications, and underlying health conditions. Chronic blepharitis, as mentioned in the context provided, is a common cause of ocular discomfort and irritation that can lead to dry eyes.\n\nWhile dry and irritated eyes are often not a sign of something serious, it is essential to monitor your symptoms and seek medical advice if they persist or worsen. In some cases, chronic dry eye syndrome can lead to complications such as corneal damage if left untreated.\n\nIf you are experiencing persistent dryness and irritation in your eyes, it is recommended to consult an eye care professional for a comprehensive eye examination. They can determine the underlying cause of your symptoms and recommend appropriate treatment options to help alleviate your discomfort.'

In [41]:
response = conversational_rag_chain.invoke(
    {"input": "I notice my vision is cloudy, especially in bright light. Could I have cataracts?"},
    config={"configurable": {"session_id": "102"}},
)
response["answer"]

'Cloudy vision, especially in bright light, can be a common symptom of cataracts. Cataracts are a condition where the lens of the eye becomes cloudy, leading to blurred or hazy vision. Other symptoms of cataracts may include difficulty seeing at night, sensitivity to light, seeing halos around lights, and faded colors.\n\nIf you are experiencing cloudy vision, particularly in bright light, it is possible that cataracts could be the cause. However, a comprehensive eye examination by an eye care professional is necessary to confirm the diagnosis. During the examination, the eye doctor will assess your symptoms, perform various tests, and evaluate the overall health of your eyes to determine if cataracts are present.\n\nIf cataracts are diagnosed, treatment options may include prescription glasses, brighter lighting, or in more advanced cases, cataract surgery to remove the cloudy lens and replace it with an artificial lens. It is important to seek professional medical advice for proper e

In [42]:
response = conversational_rag_chain.invoke(
    {"input": "What are some good habits for maintaining healthy eyes?"},
    config={"configurable": {"session_id": "102"}},
)
response["answer"]

'Maintaining healthy eyes is essential for overall well-being. Here are some good habits to help keep your eyes healthy:\n\n1. **Regular Eye Exams:** Schedule routine eye exams with an eye care professional to monitor your eye health and detect any issues early.\n\n2. **Protective Eyewear:** Wear sunglasses that block UV rays and safety glasses when engaging in activities that pose a risk to your eyes.\n\n3. **Healthy Diet:** Eat a balanced diet rich in fruits, vegetables, and omega-3 fatty acids to support eye health.\n\n4. **Stay Hydrated:** Drink an adequate amount of water to prevent dehydration, which can affect eye moisture.\n\n5. **Proper Lighting:** Ensure adequate lighting when reading or working on screens to reduce eye strain.\n\n6. **Take Breaks:** Follow the 20-20-20 rule when using digital devices – every 20 minutes, look at something 20 feet away for at least 20 seconds.\n\n7. **Avoid Smoking:** Smoking can increase the risk of eye diseases such as cataracts and macular 

In [43]:
response = conversational_rag_chain.invoke(
    {"input": "Give a list with points"},
    config={"configurable": {"session_id": "102"}},
)
response["answer"]

'Certainly! Here is a list of good habits for maintaining healthy eyes:\n\n1. Schedule regular eye exams with an eye care professional.\n2. Wear protective eyewear, such as sunglasses and safety glasses.\n3. Maintain a healthy diet rich in fruits, vegetables, and omega-3 fatty acids.\n4. Stay hydrated by drinking an adequate amount of water.\n5. Ensure proper lighting when reading or using screens to reduce eye strain.\n6. Take breaks and follow the 20-20-20 rule when using digital devices.\n7. Avoid smoking to reduce the risk of eye diseases.\n8. Manage chronic conditions like diabetes and hypertension.\n9. Practice good hygiene by washing hands before touching your eyes or handling contact lenses.\n10. Get enough quality sleep each night to allow your eyes to rest and rejuvenate.\n\nIncorporating these habits into your daily routine can help promote and maintain healthy eyes for years to come.'

In [44]:
response = conversational_rag_chain.invoke(
    {"input": "What are the symptoms of depression?"},
    config={"configurable": {"session_id": "103"}},
)
response["answer"]

"Symptoms of depression can vary from person to person, but common symptoms may include persistent feelings of sadness, hopelessness, or emptiness, loss of interest or pleasure in activities once enjoyed, changes in appetite or weight, difficulty sleeping or oversleeping, fatigue or loss of energy, feelings of worthlessness or guilt, difficulty concentrating or making decisions, and thoughts of death or suicide. It's important to note that these symptoms can also be indicative of other medical conditions, so it's essential to consult a healthcare professional for an accurate diagnosis and appropriate treatment."

In [45]:
response = conversational_rag_chain.invoke(
    {"input": "What are the symptoms of a fungal skin infection?"},
    config={"configurable": {"session_id": "104"}},
)
response["answer"]

'Symptoms of a fungal skin infection can include itching, tingling, burning sensation, and pain in the affected area. Additionally, there may be erythematous areas with a maculopapular rash present on the skin.'

In [46]:
response = conversational_rag_chain.invoke(
    {"input": "What is my name"},
    config={"configurable": {"session_id": "104"}},
)
response["answer"]

"I'm sorry, but I don't have access to your name."

In [47]:
response = conversational_rag_chain.invoke(
    {"input": "What is Deep Learning?"},
    config={"configurable": {"session_id": "105"}},
)
response["answer"]

'Deep learning is a type of machine learning that uses artificial neural networks to model and interpret complex patterns in data. It is a subset of machine learning that focuses on learning representations of data through multiple layers of interconnected nodes, known as artificial neurons. Deep learning algorithms are capable of automatically learning to recognize patterns and features from raw data, making it particularly useful for tasks such as image and speech recognition.'